# 🎓 Student Fake Data Generator

This notebook generates fake student records matching the schema below:

| Column | Description |
|---|---|
| `student_id` | Auto-incremented integer primary key |
| `student_first_name` | Randomly generated first name |
| `student_last_name` | Randomly generated last name |
| `student_email` | Derived from name in `lastname_firstname@student.ceu.edu` format |
| `student_start_year` | Random year between 2021–2025 |
| `program_id` | **Not generated yet** — see the last section to add it later |

---
**Workflow:**
1. Run **Section 1** to install/import dependencies
2. Run **Section 2** to generate the student data (without `program_id`)
3. Run **Section 3** to preview & export the data
4. Later, run **Section 4** to assign `program_id` values once you know how many programs exist

---
## Section 1 — Imports & Setup

In [ ]:
# Install Faker if not already available
# Faker is a Python library for generating realistic fake data
try:
    from faker import Faker
except ImportError:
    import subprocess
    subprocess.run(['pip', 'install', 'faker', '--quiet'], check=True)
    from faker import Faker

import pandas as pd
import random

# Seed for reproducibility — change or remove this to get different results each run
SEED = 42
random.seed(SEED)
fake = Faker()
Faker.seed(SEED)

print('✅ Setup complete.')

---
## Section 2 — Generate Student Data

Adjust `NUM_STUDENTS` to control how many rows are generated.

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────────
NUM_STUDENTS = 200       # How many student records to generate
START_YEAR_RANGE = (2021, 2025)  # Inclusive range for student_start_year
EMAIL_DOMAIN = 'student.ceu.edu' # Email domain for student accounts
# ───────────────────────────────────────────────────────────────────────────────

def generate_students(n: int) -> pd.DataFrame:
    """
    Generate n fake student records.

    Returns a DataFrame with columns:
        student_id, student_first_name, student_last_name,
        student_email, student_start_year

    NOTE: program_id is intentionally excluded — see Section 4 to add it later.
    """
    records = []

    for i in range(1, n + 1):
        first = fake.first_name()
        last  = fake.last_name()

        # Build email in the format: lastname_firstname@student.ceu.edu
        email = f"{last.lower()}_{first.lower()}@{EMAIL_DOMAIN}"

        records.append({
            'student_id':         i,
            'student_first_name': first,
            'student_last_name':  last,
            'student_email':      email,
            'student_start_year': random.randint(*START_YEAR_RANGE),
        })

    return pd.DataFrame(records)


# Generate the data — stored in `df` for use in later cells
df = generate_students(NUM_STUDENTS)

print(f'✅ Generated {len(df)} student records.')

---
## Section 3 — Preview & Export

In [ ]:
# Preview the first 10 rows
print(f'Shape: {df.shape}  |  Columns: {list(df.columns)}\n')
df.head(10)

In [ ]:
# Export to CSV (without program_id — it will be added in Section 4)
OUTPUT_PATH = 'students_no_program.csv'

df.to_csv(OUTPUT_PATH, index=False)
print(f'✅ Saved to "{OUTPUT_PATH}"')

---
## Section 4 — Add `program_id` (run this later)

Once you know how many programs exist, come back here and fill in `NUM_PROGRAMS` below.

This section will:
- Randomly assign each student a `program_id` from `1` to `NUM_PROGRAMS`
- Insert the column in the correct position (after `student_start_year`)
- Preview the updated DataFrame
- Save a new CSV that matches the original schema exactly

> ⚠️ **Make sure Section 2 has been run first** so that `df` exists in memory.
> If you've restarted the kernel, re-run Sections 1–3 before running this cell.

In [ ]:
import pandas as pd
import random

In [ ]:
# ── ✏️  FILL THIS IN when you're ready ─────────────────────────────────────────
NUM_PROGRAMS = 79  # <-- Replace None with the number of programs, e.g. 5
# ───────────────────────────────────────────────────────────────────────────────

if NUM_PROGRAMS is None:
    print('⚠️  Set NUM_PROGRAMS to the number of programs and re-run this cell.')

else:
    # Sanity check
    if not isinstance(NUM_PROGRAMS, int) or NUM_PROGRAMS < 1:
        raise ValueError('NUM_PROGRAMS must be a positive integer.')

    # Load from the saved CSV so this section is self-contained
    # (safe to run even if df is no longer in memory after a kernel restart)
    df_with_program = pd.read_csv('students_no_program.csv')

    # Randomly assign a program_id between 1 and NUM_PROGRAMS to every student
    df_with_program['program_id'] = [
        random.randint(1, NUM_PROGRAMS) for _ in range(len(df_with_program))
    ]

    # Reorder columns to match the original schema exactly
    df_with_program = df_with_program[[
        'student_id',
        'student_first_name',
        'student_last_name',
        'student_email',
        'student_start_year',
        'program_id',
    ]]

    # Preview distribution of program assignments
    print(f'✅ Assigned program_id (1–{NUM_PROGRAMS}) to {len(df_with_program)} students.\n')
    print('Program distribution:')
    print(df_with_program['program_id'].value_counts().sort_index().to_string())
    print()

    # Save the final complete CSV
    FINAL_OUTPUT_PATH = 'students_complete.csv'
    df_with_program.to_csv(FINAL_OUTPUT_PATH, index=False)
    print(f'✅ Saved complete data to "{FINAL_OUTPUT_PATH}"')

    # Show a preview
    df_with_program.head(10)